# Лабораторная работа #9: Введение в NLP

## Импорт библиотек

In [3]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import re
from datasets import load_dataset
import pandas as pd
import spacy
from nltk.stem.snowball import SnowballStemmer
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import re

import warnings
warnings.filterwarnings("ignore")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Базовые методы работы с текстом

Данная глава должна наглядно показать, как обработать простейший текст с помощью разных методов

Первый доступный нам способ обработки строк – посимвольная замена. Вы её уже видели много раз (в том числе на ЕГЭ, там str.replace точно применяется). Следующий этап – [регулярные выражения](https://ru.wikipedia.org/wiki/%D0%A0%D0%B5%D0%B3%D1%83%D0%BB%D1%8F%D1%80%D0%BD%D1%8B%D0%B5_%D0%B2%D1%8B%D1%80%D0%B0%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F). Её и используем для удаления лишних символов (не букв, знаков пунктуации в нашем случае), а также лишних пробелов:

In [4]:
def clean_text(t):
    t = t.lower()
    t = re.sub(r"<.*?>", " ", t)
    t = re.sub(r"[^a-z ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


Можно пойти дальше, и удалить целые слова. Действительно, некоторые слова служат исключительно для связки и не несут никакой семантики, болтаясь балластом. Это так называемые "стоп-слова": разнообразные артикли, частицы, вводные слова и союзы. Их список можно достать средствами nltk.stopwords. Дальше их очень просто удалить. Одна проблема: нам нужна не строка, а список слов. Об этом пойдёт речь дальше.

In [5]:
def remove_stopwords(t):
  filtered_words = []
  for word in t:
    if word in set(stopwords.words('english')):
        continue
    filtered_words.append(word)
  return filtered_words

Продолжаем рост сложности. Теперь используем библиотеку spaCy для токенизации и лемматизации, а также SnowBallStemmer из библиотеки nltk для стемминга.

Краткое напоминание:
1. Токенизация – разбиение текста на слова, предложения или другие единицы (токены). Грубо говоря, строку делим на массив строк (как метод split).

2. Лемматизация – приведение слова к словарной форме (лемме) с учётом контекста и морфологии (`fading -> fade`)

3. Стемминг - грубое отсечение окончаний и суффиксов для приведения слова к базовой форме (`location -> locat`).

Подробнее можно было узнать на лекции... ну или в интернете: [Токенизация](https://ru.wikipedia.org/wiki/%D0%9B%D0%B5%D0%BA%D1%81%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7), [Лемматизация](https://ru.wikipedia.org/wiki/%D0%9B%D0%B5%D0%BC%D0%BC%D0%B0%D1%82%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D1%8F), [Стемминг](https://ru.wikipedia.org/wiki/%D0%A1%D1%82%D0%B5%D0%BC%D0%BC%D0%B8%D0%BD%D0%B3).

In [6]:
nlp = spacy.load("en_core_web_sm")
stemmer = SnowballStemmer("english")

В качестве примера возьмём выборку из 5 кратких текстов на английском языке. Тексты взяты из датасета [UFO Sightings](https://www.kaggle.com/datasets/NUFORC/ufo-sightings/data)

In [7]:
texts = [
  "Bright white light&#44 dissapearing and reappearing in different locations in the night sky&#44 eventually dissapearing&#44 fading away.",
  "Six orange orbs plague the skies of Publix in Davie.",
  "Strange noises",
  "SAW TWO SMALL 3-4 FOOT TRIANGULAR SHAPED DULL METALIC OBJECTS BELOW THE CLOUDS MOVING IN A SYNCRONIZED MANNER  ROLLING END OVER END AN",
  "Glowing moon-sized object moves across the sky&#44 only to leave an orange streak behind it before it dissapears."
]
texts

['Bright white light&#44 dissapearing and reappearing in different locations in the night sky&#44 eventually dissapearing&#44 fading away.',
 'Six orange orbs plague the skies of Publix in Davie.',
 'Strange noises',
 'SAW TWO SMALL 3-4 FOOT TRIANGULAR SHAPED DULL METALIC OBJECTS BELOW THE CLOUDS MOVING IN A SYNCRONIZED MANNER  ROLLING END OVER END AN',
 'Glowing moon-sized object moves across the sky&#44 only to leave an orange streak behind it before it dissapears.']

Возьмём первый текст. Почистим его регуляркой:

In [8]:
text = clean_text(texts[0])
text

'bright white light dissapearing and reappearing in different locations in the night sky eventually dissapearing fading away'

Вытащим информацию о токене, лемму и стем:

In [9]:
document = nlp(text)
for token in document:
  print(f"TOKEN: {token.text}; LEMMA: {token.lemma_}; STEM: {stemmer.stem(token.lemma_)}")

TOKEN: bright; LEMMA: bright; STEM: bright
TOKEN: white; LEMMA: white; STEM: white
TOKEN: light; LEMMA: light; STEM: light
TOKEN: dissapearing; LEMMA: dissapeare; STEM: dissapear
TOKEN: and; LEMMA: and; STEM: and
TOKEN: reappearing; LEMMA: reappear; STEM: reappear
TOKEN: in; LEMMA: in; STEM: in
TOKEN: different; LEMMA: different; STEM: differ
TOKEN: locations; LEMMA: location; STEM: locat
TOKEN: in; LEMMA: in; STEM: in
TOKEN: the; LEMMA: the; STEM: the
TOKEN: night; LEMMA: night; STEM: night
TOKEN: sky; LEMMA: sky; STEM: sky
TOKEN: eventually; LEMMA: eventually; STEM: eventu
TOKEN: dissapearing; LEMMA: dissapeare; STEM: dissapear
TOKEN: fading; LEMMA: fade; STEM: fade
TOKEN: away; LEMMA: away; STEM: away


Соберём стемы в отдельный список:

In [10]:
removed_stopwords = remove_stopwords([token.text for token in document]) # beware, now this is a list of strings, not tokens!!!
final_tokens = [stemmer.stem(word) for word in removed_stopwords]
result = " ".join(final_tokens)
result

'bright white light dissapear reappear differ locat night sky eventu dissapear fade away'

In [11]:
final_tokens_array = []
final_tokens_array.append(result)
# TODO: Токенизировать, удалить стоп-слова, провести лемматизацию и стемминг других 4-х строк из массива texts
for i in range(1, 5):
  text = clean_text(texts[i])
  d = nlp(text)
  for token in d:
      print(f"TOKEN: {token.text}; LEMMA: {token.lemma_}; STEM: {stemmer.stem(token.lemma_)}")
  print('\n')
  removed_stopwords = remove_stopwords([token.text for token in d])
  final_tokens = [stemmer.stem(word) for word in removed_stopwords]
  result = " ".join(final_tokens)
  final_tokens_array.append(result)
  print(result)
  print('\n')



TOKEN: six; LEMMA: six; STEM: six
TOKEN: orange; LEMMA: orange; STEM: orang
TOKEN: orbs; LEMMA: orbs; STEM: orb
TOKEN: plague; LEMMA: plague; STEM: plagu
TOKEN: the; LEMMA: the; STEM: the
TOKEN: skies; LEMMA: sky; STEM: sky
TOKEN: of; LEMMA: of; STEM: of
TOKEN: publix; LEMMA: publix; STEM: publix
TOKEN: in; LEMMA: in; STEM: in
TOKEN: davie; LEMMA: davie; STEM: davi


six orang orb plagu sky publix davi


TOKEN: strange; LEMMA: strange; STEM: strang
TOKEN: noises; LEMMA: noise; STEM: nois


strang nois


TOKEN: saw; LEMMA: see; STEM: see
TOKEN: two; LEMMA: two; STEM: two
TOKEN: small; LEMMA: small; STEM: small
TOKEN: foot; LEMMA: foot; STEM: foot
TOKEN: triangular; LEMMA: triangular; STEM: triangular
TOKEN: shaped; LEMMA: shape; STEM: shape
TOKEN: dull; LEMMA: dull; STEM: dull
TOKEN: metalic; LEMMA: metalic; STEM: metal
TOKEN: objects; LEMMA: object; STEM: object
TOKEN: below; LEMMA: below; STEM: below
TOKEN: the; LEMMA: the; STEM: the
TOKEN: clouds; LEMMA: cloud; STEM: cloud
TOKEN: mov

Теперь попытаемся преобразовать полученный список токенов к численному виду (так называемый Embedding, математический вектор чисел). Есть множество подходов к решению этой задачи. Выделим два из них: Bag-of-words и TF-IDF. Краткое пояснение к методам:
1. Bag-of-words. Массив слов преобразуется в массив частот слов. Размер массива такой, сколько слов в тексте (или в целевых данных, можно хоть все слова русского языка брать, тогда будет массив длины 100 тыс. и больше). Прост для понимания, но размерность получаемого пространства ужасающе огромная
2. TF-IDF. Вместо частоты встречи слова считает метрики TF (Term Frequency, частота слова в текущем документе) и IDF (Inverse Document Frequency, обратная частота слова среди всех документов). Они перемножаются, и получается, что слова, которые во всех текстах встречаются, получают маленький вес, а редкие и специфичные для одного документа – большой. Отсекая слова с маленьким весом, можно значительно уменьшить размерность пространства.

Подробнее было на лекции, или в интернете: [Bag-of-words](https://en.wikipedia.org/wiki/Bag-of-words_model), [TF-IDF](https://ru.wikipedia.org/wiki/TF-IDF).

Метод Bag-of-Words реализуем с помощью CountVectorizer из sklearn (он нас теперь преследует), TF-IDF с помощью TfidfVectorizer оттуда же.

Будьте осторожны: CountVectorizer и TfidfVectorizer принимают на вход в fit_transform набор строк, а не набор массивов строк. Поэтому финальные токены придётся join-ить через пробел...

In [12]:
cvec = CountVectorizer(max_features=20000)
final_tokens_cnt = cvec.fit_transform([' '.join(final_tokens)])
cvec.vocabulary_

{'glow': np.int64(3),
 'moon': np.int64(5),
 'size': np.int64(9),
 'object': np.int64(7),
 'move': np.int64(6),
 'across': np.int64(0),
 'sky': np.int64(10),
 'leav': np.int64(4),
 'orang': np.int64(8),
 'streak': np.int64(11),
 'behind': np.int64(1),
 'dissapear': np.int64(2)}

In [13]:
print(final_tokens_cnt)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 12 stored elements and shape (1, 12)>
  Coords	Values
  (0, 3)	1
  (0, 5)	1
  (0, 9)	1
  (0, 7)	1
  (0, 6)	1
  (0, 0)	1
  (0, 10)	1
  (0, 4)	1
  (0, 8)	1
  (0, 11)	1
  (0, 1)	1
  (0, 2)	1


Как это прочитать: во втором принте у нас значения: в CountVectorizer частота, в TF-IDF - значение TF*IDF. Из координат берёте второе значение, ищете его в словаре, оно соответствует слову.

In [14]:
tfidf = TfidfVectorizer(max_features=20000)
final_tokens_tfidf = tfidf.fit_transform([' '.join(final_tokens)])
tfidf.vocabulary_

{'glow': np.int64(3),
 'moon': np.int64(5),
 'size': np.int64(9),
 'object': np.int64(7),
 'move': np.int64(6),
 'across': np.int64(0),
 'sky': np.int64(10),
 'leav': np.int64(4),
 'orang': np.int64(8),
 'streak': np.int64(11),
 'behind': np.int64(1),
 'dissapear': np.int64(2)}

In [33]:
print(final_tokens_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12 stored elements and shape (1, 12)>
  Coords	Values
  (0, 3)	0.2886751345948129
  (0, 5)	0.2886751345948129
  (0, 9)	0.2886751345948129
  (0, 7)	0.2886751345948129
  (0, 6)	0.2886751345948129
  (0, 0)	0.2886751345948129
  (0, 10)	0.2886751345948129
  (0, 4)	0.2886751345948129
  (0, 8)	0.2886751345948129
  (0, 11)	0.2886751345948129
  (0, 1)	0.2886751345948129
  (0, 2)	0.2886751345948129


## Практическое задание

Работать нужно будет с датасетом отзывов IMDB. Он содержит два столбца: текст отзыва и бинарная оценка (положительный/отрицательный). Задача: по тексту отзыва предсказать, положительный он или отрицательный.

Логика, соответствующая обучению классификаторов, уже написана за вас.

От вас требуется предобработать столбец с отзывами так, чтобы он превратился из текста в численный (каждому отзыву сопоставить Embedding, вектор чисел). Следует сделать это двумя способами: CountVectorizer и TF-IDF. Не забудьте про токенизацию, лемматизацию, стемминг. Функциями для обработки пробелов и не-букв регулярками, а также удаление стоп-слов можно подхватить из примеров выше.

In [16]:
NSAMPLES = 100

In [22]:
ds = load_dataset("stanfordnlp/imdb") # сохраняем в переменную наш датасет. Можно прям в поисковик вставить, там слева сверху можно посмотреть и train и test (это два разных датасета)
ds = pd.concat([pd.DataFrame(ds["train"]), pd.DataFrame(ds["test"])]).sample(NSAMPLES) # объединяем train и test в один датасет с помощью pd.concat. sample это сколько рандомно выбраных строк туда войдет
X, y = ds["text"], ds["label"] # X - текст, y - оценка
ds

,text,label
10889,"This movie goes beyond just being bad, it is d...",0
24936,"What are the odds of a ""Mermaid"" helium balloo...",1
8761,I waited quite awhile till I was able to watch...,0
24182,This tender beautifully crafted production del...,1
14998,"You can't watch a film like Peter Watkins' ""Pr...",1
23002,"""The Piano Teacher"" is all about Huppert's cha...",1
7425,"OK, so I am an original Wicker Man fan and I u...",0
15979,Joseph L. Mankiewicz is not remembered by most...,1
3307,I absolutely hate it when a film completely fa...,0
7715,This is one lowly film. It has no real plot. W...,0


In [37]:
def preprocess_text(text):    # Предобработка текста
    cleaned_text = clean_text(text)     # Очистка текста

    doc = nlp(cleaned_text)     # Токенизация с помощью spaCy

    # Удаление стоп-слов и стемминг. Кроме стема нам ничего не надо
    tokens = [token.text for token in doc]
    filtered_tokens = remove_stopwords(tokens)
    stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]

    return " ".join(stemmed_tokens)     # Объединение обратно в строку

processed_texts = [preprocess_text(text) for text in X] # применяем нашу функцию на текст и заносим в список

X_train, X_test, y_train, y_test = train_test_split(processed_texts, y, test_size=0.2, random_state=42) # разбиваем на трейн и тест (об этом вся прошлая лаба)

# Дальше мы по сути повторяем то, что уже делали в этой лабе. Гораздо сложнее чем кажется
# Создаем векторизаторы
count_vectorizer = CountVectorizer(max_features=20000)
tfidf_vectorizer = TfidfVectorizer(max_features=20000)

# Векторизуем данные с помощью CountVectorizer
X_train_countvect = count_vectorizer.fit_transform(X_train)
X_test_countvect = count_vectorizer.transform(X_test)

# Векторизуем данные с помощью TF-IDF
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Размер обучающей выборки: {X_train_countvect.shape}")
print(f"Размер тестовой выборки: {X_test_countvect.shape}")

Размер обучающей выборки: (16, 939)
Размер тестовой выборки: (4, 939)


In [42]:
# Это как было, и это не правильно, оно вызывает ошибу и аневризму у меня в башке. Чтото не так с импортом
# model = LogisticRegression(max_iter=200)
# model.fit(X_train_countvect, y_train)
# y_pred = clf_count.predict(X_test_countvect)
# print("CountVectorizer Accuracy:", accuracy_score(y_test, y_pred))

# >>> NameError                                 Traceback (most recent call last)
# >>> /tmp/ipython-input-375673727.py in <cell line: 0>()
# >>>       1 model = LogisticRegression(max_iter=200)
# >>>       2 model.fit(X_train_countvect, y_train)
# >>> ----> 3 y_pred = clf_count.predict(X_test_countvect)
# >>>       4 print("CountVectorizer Accuracy:", accuracy_score(y_test, y_pred))

# >>> NameError: name 'clf_count' is not defined


# Это моё

# Модель с CountVectorizer
model_count = LogisticRegression(max_iter=200) # Создает алгоритм машинного обучения для классификации с максимальным количеством итераций 200
model_count.fit(X_train_countvect, y_train) # Обучает модель на тренировочных данных
y_pred_count = model_count.predict(X_test_countvect) # Делает предсказания на тестовых данных


In [43]:
# Опять же пришлоь переписать

print("CountVectorizer Accuracy:", accuracy_score(y_test, y_pred_count)) # Вычисляет и выводит точность модели по фермуле (количество правильных предсказаний) / (общее количество примеров)
print("Bag-of-words Classification Report:")
print(classification_report(y_test, y_pred_count, target_names=["neg", "pos"]))
# Precision (Точность): Из всех предсказанных "позитивных", сколько действительно позитивные
# Recall (Полнота): Из всех реальных "позитивных", сколько мы правильно нашли
# F1-score: Гармоническое среднее между precision и recall
# Support: Количество примеров в каждом классе

CountVectorizer Accuracy: 0.5
Bag-of-words Classification Report:
              precision    recall  f1-score   support

         neg       0.00      0.00      0.00         2
         pos       0.50      1.00      0.67         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4



In [44]:
# Не работает, ошибка таже самая

# model = LogisticRegression(max_iter=200)
# model.fit(X_train_tfidf, y_train)
# pred_tfidf = clf_tfidf.predict(X_test_tfidf)
# print("TfidfVectorizer Accuracy:", accuracy_score(y_test, pred_tfidf))


# Это моё

# Модель с TF-IDF
model_tfidf = LogisticRegression(max_iter=200)
model_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)


In [45]:
# Опять переписывал

print("TfidfVectorizer Accuracy:", accuracy_score(y_test, y_pred_tfidf))
print("TF-IDF Classification Report:")
print(classification_report(y_test, y_pred_tfidf, target_names=["neg", "pos"]))


TfidfVectorizer Accuracy: 0.5
TF-IDF Classification Report:
              precision    recall  f1-score   support

         neg       0.00      0.00      0.00         2
         pos       0.50      1.00      0.67         2

    accuracy                           0.50         4
   macro avg       0.25      0.50      0.33         4
weighted avg       0.25      0.50      0.33         4

